# MSD Special State With Noiseless Prefix And Noisy Suffix

This notebook prototypes the split you described:

- the tomography-inverse gate on the measured input qubit and the 5-qubit `MSD` inverse are kept **noiseless**
- the Steane encoding, logical `MSD`, and output tomography are written as a **noisy physical squin suffix**
- the resulting physical squin kernels are plugged into `GeminiLogicalSimulatorTask`, so we can still sample detector/observable shots and build a detector error model

This version keeps the same noiseless behavior as the earlier hybrid notebook: detectors are all `0`, and the observables are deterministic.

Important caveat: this is a **custom physical squin prototype**. It uses the Gemini noise-model hooks directly for the suffix, but it does **not** yet splice into the exact compiler-emitted noisy physical squin kernel, so compiler-inserted move/lane/idle noise is not preserved here.


In [2]:
from IPython.display import HTML, display
from pathlib import Path
import sys

import numpy as np
from kirin.dialects import ilist

from bloqade import qubit, squin
from bloqade.gemini import logical as gemini_logical
from bloqade.gemini.logical.stdlib import default_post_processing
from bloqade.lanes import GeminiLogicalSimulator
from bloqade.tsim import Circuit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "demo").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from demo.msd_utils.circuits import build_task
from demo.msd_utils.core import run_task

BASIS_LABELS = ("X", "Y", "Z")


In [3]:
def show_diagram(diagram_html: str, *, height: int = 360) -> None:
    display(
        HTML(
            f'<div style="overflow-x:auto; width:100%; border:1px solid #ddd; padding:8px; background:white; max-height:{height}px;">'
            + diagram_html
            + "</div>"
        )
    )


## Logical Suffix

We still define the logical `MSD` suffix in the Gemini logical dialect. We use these logical tasks as the anchor for post-processing and for the detector error model interface, but we override their `physical_squin_kernel` with our custom hybrid physical kernels below.


In [4]:
@squin.kernel
def logical_msd_forward(reg):
    squin.broadcast.sqrt_x([reg[0], reg[1], reg[4]])
    squin.broadcast.cz([reg[0], reg[2]], [reg[1], reg[3]])
    squin.broadcast.sqrt_y([reg[0], reg[3]])
    squin.broadcast.cz([reg[0], reg[3]], [reg[2], reg[4]])
    squin.sqrt_x_adj(reg[0])
    squin.broadcast.cz([reg[0], reg[1]], [reg[4], reg[3]])
    squin.broadcast.sqrt_x_adj(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_x():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_y():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    squin.sqrt_z_adj(reg[0])
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_z():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    return default_post_processing(reg)


sim = GeminiLogicalSimulator()
logical_suffix_kernels = {
    "X": logical_suffix_x,
    "Y": logical_suffix_y,
    "Z": logical_suffix_z,
}

logical_tasks = {
    basis: build_task(sim, kernel, m2dets=None, m2obs=None, append_measurements=False)
    for basis, kernel in logical_suffix_kernels.items()
}

{basis: (task.detector_error_model.num_detectors, task.detector_error_model.num_observables) for basis, task in logical_tasks.items()}


{'X': (15, 5), 'Y': (15, 5), 'Z': (15, 5)}

## Noiseless Prefix, Noisy Physical Suffix

The inverse prefix is kept noiseless. The encoding, logical `MSD`, and output tomography are written directly as noisy physical squin helpers using the Gemini noise model.


In [5]:
@squin.kernel
def physical_msd_inverse(inputs):
    squin.broadcast.sqrt_x(inputs)
    squin.broadcast.cz([inputs[0], inputs[1]], [inputs[4], inputs[3]])
    squin.sqrt_x(inputs[0])
    squin.broadcast.cz([inputs[0], inputs[3]], [inputs[2], inputs[4]])
    squin.broadcast.sqrt_y_adj([inputs[0], inputs[3]])
    squin.broadcast.cz([inputs[0], inputs[2]], [inputs[1], inputs[3]])
    squin.broadcast.sqrt_x_adj([inputs[0], inputs[1], inputs[4]])


local_r_noise = sim.noise_model.local_r_noise
local_rz_noise = sim.noise_model.local_rz_noise
global_r_noise = sim.noise_model.global_r_noise
cz_paired_noise = sim.noise_model.cz_paired_noise
cz_unpaired_noise = sim.noise_model.cz_unpaired_noise

x_axis = 0.0
y_axis = 0.25
half_turn = 0.5
quarter_turn = 0.25


@squin.kernel
def steane7_encode_existing_noisy(qubits):
    evens = qubits[::2]
    odds = qubits[1::2]

    first6 = qubits[:6]
    squin.broadcast.sqrt_y_adj(first6)
    local_r_noise(first6, y_axis, -quarter_turn)

    targets1 = evens[1:]
    squin.broadcast.cz(odds, targets1)
    cz_paired_noise(odds, targets1)
    cz_unpaired_noise(ilist.IList([qubits[0]]))

    squin.sqrt_y(qubits[6])
    local_r_noise(ilist.IList([qubits[6]]), y_axis, quarter_turn)

    controls2 = evens[:-1]
    targets2 = ilist.IList([qubits[3], qubits[5], qubits[6]])
    squin.broadcast.cz(controls2, targets2)
    cz_paired_noise(controls2, targets2)
    cz_unpaired_noise(ilist.IList([qubits[1]]))

    tail = qubits[2:]
    squin.broadcast.sqrt_y(tail)
    local_r_noise(tail, y_axis, quarter_turn)

    controls3 = evens[:-1]
    targets3 = odds
    squin.broadcast.cz(controls3, targets3)
    cz_paired_noise(controls3, targets3)
    cz_unpaired_noise(ilist.IList([qubits[6]]))

    subset = ilist.IList([qubits[1], qubits[2], qubits[4]])
    squin.broadcast.sqrt_y(subset)
    local_r_noise(subset, y_axis, quarter_turn)

    squin.x(qubits[3])
    local_r_noise(ilist.IList([qubits[3]]), x_axis, half_turn)

    z_subset = ilist.IList([qubits[1], qubits[5]])
    squin.broadcast.z(z_subset)
    local_rz_noise(z_subset, half_turn)


@squin.kernel
def noisy_transversal_logical_msd(q0, q1, q2, q3, q4):
    first = q0 + q1 + q4
    squin.broadcast.sqrt_x(first)
    local_r_noise(first, x_axis, quarter_turn)

    controls1 = q0 + q2
    targets1 = q1 + q3
    squin.broadcast.cz(controls1, targets1)
    cz_paired_noise(controls1, targets1)
    cz_unpaired_noise(q4)

    second = q0 + q3
    squin.broadcast.sqrt_y(second)
    local_r_noise(second, y_axis, quarter_turn)

    controls2 = q0 + q3
    targets2 = q2 + q4
    squin.broadcast.cz(controls2, targets2)
    cz_paired_noise(controls2, targets2)
    cz_unpaired_noise(q1)

    squin.broadcast.sqrt_x_adj(q0)
    local_r_noise(q0, x_axis, -quarter_turn)

    controls3 = q0 + q1
    targets3 = q4 + q3
    squin.broadcast.cz(controls3, targets3)
    cz_paired_noise(controls3, targets3)
    cz_unpaired_noise(q2)

    all_qubits = q0 + q1 + q2 + q3 + q4
    squin.broadcast.sqrt_x_adj(all_qubits)
    global_r_noise(all_qubits, x_axis, -quarter_turn)


@squin.kernel
def noisy_output_tomography_x(q0):
    squin.broadcast.sqrt_z(q0)
    local_rz_noise(q0, quarter_turn)
    squin.broadcast.sqrt_x(q0)
    local_r_noise(q0, x_axis, quarter_turn)
    squin.broadcast.sqrt_z(q0)
    local_rz_noise(q0, quarter_turn)


@squin.kernel
def noisy_output_tomography_y(q0):
    squin.broadcast.sqrt_z_adj(q0)
    local_rz_noise(q0, -quarter_turn)
    squin.broadcast.sqrt_z(q0)
    local_rz_noise(q0, quarter_turn)
    squin.broadcast.sqrt_x(q0)
    local_r_noise(q0, x_axis, quarter_turn)
    squin.broadcast.sqrt_z(q0)
    local_rz_noise(q0, quarter_turn)


@squin.kernel
def noisy_output_tomography_z(q0):
    return


@squin.kernel
def annotate_hybrid_measurements(m):
    squin.set_detector([m[0], m[1], m[2], m[3]], coordinates=[0])
    squin.set_detector([m[1], m[2], m[4], m[5]], coordinates=[1])
    squin.set_detector([m[2], m[3], m[4], m[6]], coordinates=[2])
    squin.set_detector([m[7], m[8], m[9], m[10]], coordinates=[3])
    squin.set_detector([m[8], m[9], m[11], m[12]], coordinates=[4])
    squin.set_detector([m[9], m[10], m[11], m[13]], coordinates=[5])
    squin.set_detector([m[14], m[15], m[16], m[17]], coordinates=[6])
    squin.set_detector([m[15], m[16], m[18], m[19]], coordinates=[7])
    squin.set_detector([m[16], m[17], m[18], m[20]], coordinates=[8])
    squin.set_detector([m[21], m[22], m[23], m[24]], coordinates=[9])
    squin.set_detector([m[22], m[23], m[25], m[26]], coordinates=[10])
    squin.set_detector([m[23], m[24], m[25], m[27]], coordinates=[11])
    squin.set_detector([m[28], m[29], m[30], m[31]], coordinates=[12])
    squin.set_detector([m[29], m[30], m[32], m[33]], coordinates=[13])
    squin.set_detector([m[30], m[31], m[32], m[34]], coordinates=[14])
    squin.set_observable([m[0], m[1], m[5]])
    squin.set_observable([m[7], m[8], m[12]])
    squin.set_observable([m[14], m[15], m[19]])
    squin.set_observable([m[21], m[22], m[26]])
    squin.set_observable([m[28], m[29], m[33]])


In [6]:
@squin.kernel
def hybrid_special_x():
    q = squin.qalloc(35)
    squin.broadcast.reset(q)
    q0 = q[0:7]
    q1 = q[7:14]
    q2 = q[14:21]
    q3 = q[21:28]
    q4 = q[28:35]
    inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

    squin.h(q0[6])
    physical_msd_inverse(inputs)
    steane7_encode_existing_noisy(q0)
    steane7_encode_existing_noisy(q1)
    steane7_encode_existing_noisy(q2)
    steane7_encode_existing_noisy(q3)
    steane7_encode_existing_noisy(q4)
    noisy_transversal_logical_msd(q0, q1, q2, q3, q4)
    noisy_output_tomography_x(q0)
    m = squin.broadcast.measure(q)
    annotate_hybrid_measurements(m)


@squin.kernel
def hybrid_special_y():
    q = squin.qalloc(35)
    squin.broadcast.reset(q)
    q0 = q[0:7]
    q1 = q[7:14]
    q2 = q[14:21]
    q3 = q[21:28]
    q4 = q[28:35]
    inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

    squin.h(q0[6])
    squin.sqrt_z(q0[6])
    physical_msd_inverse(inputs)
    steane7_encode_existing_noisy(q0)
    steane7_encode_existing_noisy(q1)
    steane7_encode_existing_noisy(q2)
    steane7_encode_existing_noisy(q3)
    steane7_encode_existing_noisy(q4)
    noisy_transversal_logical_msd(q0, q1, q2, q3, q4)
    noisy_output_tomography_y(q0)
    m = squin.broadcast.measure(q)
    annotate_hybrid_measurements(m)


@squin.kernel
def hybrid_special_z():
    q = squin.qalloc(35)
    squin.broadcast.reset(q)
    q0 = q[0:7]
    q1 = q[7:14]
    q2 = q[14:21]
    q3 = q[21:28]
    q4 = q[28:35]
    inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

    physical_msd_inverse(inputs)
    steane7_encode_existing_noisy(q0)
    steane7_encode_existing_noisy(q1)
    steane7_encode_existing_noisy(q2)
    steane7_encode_existing_noisy(q3)
    steane7_encode_existing_noisy(q4)
    noisy_transversal_logical_msd(q0, q1, q2, q3, q4)
    noisy_output_tomography_z(q0)
    m = squin.broadcast.measure(q)
    annotate_hybrid_measurements(m)


hybrid_physical_kernels = {
    "X": hybrid_special_x,
    "Y": hybrid_special_y,
    "Z": hybrid_special_z,
}

hybrid_tasks = {}
for basis in BASIS_LABELS:
    task = build_task(sim, logical_suffix_kernels[basis], m2dets=None, m2obs=None, append_measurements=False)
    task.__dict__["physical_squin_kernel"] = hybrid_physical_kernels[basis]
    hybrid_tasks[basis] = task

{basis: (task.detector_error_model.num_detectors, task.detector_error_model.num_observables) for basis, task in hybrid_tasks.items()}


{'X': (15, 5), 'Y': (15, 5), 'Z': (15, 5)}

## Check The Noiseless Reference And The Detector Error Model


In [7]:
def summarize_task(task, *, shots: int = 256) -> dict[str, object]:
    noiseless = run_task(task, shots, with_noise=False)
    noisy = run_task(task, shots, with_noise=True)
    return {
        "noiseless_all_zero_detectors": bool(np.all(noiseless.detectors == 0)),
        "noiseless_unique_observables": np.unique(noiseless.observables, axis=0).tolist(),
        "num_detectors": task.detector_error_model.num_detectors,
        "num_observables": task.detector_error_model.num_observables,
        "noisy_detector_shape": tuple(noisy.detectors.shape),
        "noisy_observable_shape": tuple(noisy.observables.shape),
    }


HYBRID_NOISE_SPLIT_RESULTS = {
    basis: summarize_task(task)
    for basis, task in hybrid_tasks.items()
}

HYBRID_NOISE_SPLIT_RESULTS


{'X': {'noiseless_all_zero_detectors': True,
  'noiseless_unique_observables': [[0, 1, 0, 1, 1]],
  'num_detectors': 15,
  'num_observables': 5,
  'noisy_detector_shape': (256, 15),
  'noisy_observable_shape': (256, 5)},
 'Y': {'noiseless_all_zero_detectors': True,
  'noiseless_unique_observables': [[1, 1, 0, 1, 1]],
  'num_detectors': 15,
  'num_observables': 5,
  'noisy_detector_shape': (256, 15),
  'noisy_observable_shape': (256, 5)},
 'Z': {'noiseless_all_zero_detectors': True,
  'noiseless_unique_observables': [[0, 1, 0, 1, 1]],
  'num_detectors': 15,
  'num_observables': 5,
  'noisy_detector_shape': (256, 15),
  'noisy_observable_shape': (256, 5)}}

## Visualize One Basis

This is the actual physical squin kernel we hand to the task for the `Z` basis. It includes inline detector and observable annotations, so the same kernel is the source of the detector error model.


In [8]:
show_diagram(str(Circuit(hybrid_physical_kernels["Z"]).diagram(height=480)), height=480)


## Notes

- The prefix is truly noiseless in this prototype: we do not call any noise hooks during the tomography-inverse prep or the 5-qubit `MSD` inverse.
- The suffix is noisy and detector-annotated, so `hybrid_tasks[basis].detector_error_model` works end-to-end.
- In noiseless simulation the detectors are all `0`, and the observables are deterministic but not forced to `00000`; this matches the earlier hybrid-notebook behavior.
